# FastConformer → LiteRT / ONNX: export and validation (Colab)

Exports `obadx/muaalem-fastconformer-base-v1` to LiteRT (`.tflite`) and ONNX, in fp32 and int8,
and checks the result against the PyTorch model **on the real checkpoint**. Details are in
`docs/fastconformer_export.md`.

| Step | What it checks | Plan gate |
|---|---|---|
| Unit tests | Mel graph vs real NeMo; split graphs vs the PyTorch streaming path; guards; no Flex ops; mel and pre-encoder free of int8 | 1, 2, 3, 6, 7 |
| Export | Builds 4 graphs × 2 formats × fp32/int8 and reports sizes, Flex ops, and XNNPACK partitions | 6, 7 |
| Fidelity | Per-utterance logits of every bundle vs PyTorch: fp32 error ≤ 1e-3 with identical argmax | 4 |
| PER | Phoneme PER on QDAT (159 clips, full) and a test-set subset, per bundle vs PyTorch: fp32 Δ ≤ 2e-4, int8 QDAT Δ ≤ 0.005 | 5 |

## Before you run

1. **Hugging Face token.** It needs read access to `obadx/muaalem-fastconformer-base-v1` (private) and,
   for the test-set PER, `obadx/muaalem-annotated-v3`. Add it in the 🔑 **Secrets** sidebar as `HF_TOKEN`
   (or `HF_TOKEN_MUALEM`, as in the mic notebook), and switch on **Notebook access**.
2. **Runtime.** Runtime → Change runtime type → **CPU**, with **High-RAM** if offered. No GPU is needed:
   the exported graphs run on CPU, and so does the reference here so that both are compared on the same device.
3. **Google Drive** (recommended). Allow the mount when asked. Exports (~1.1 GB) and results then survive a
   disconnect, and the PER step resumes where it stopped.
4. **Runtime → Run all.**

Rough times on a standard CPU runtime: install ~10 min, tests ~10 min, export ~10–15 min, fidelity
~20–40 min, PER the longest (QDAT runs 5 times; the test-set subset adds more). At the end, send back the
`validation` folder (or just the two `report_*.json` files) that the last cell points to.

In [ ]:
# Settings
MODEL_ID = "obadx/muaalem-fastconformer-base-v1"
BRANCH = "qunatized-fastconformer"
REPO_URL = "https://github.com/Abdullah-Haytham/prepare-quran-dataset.git"

USE_DRIVE = True          # keep exports and results in Google Drive
NUM_QDAT_FIDELITY = 100   # QDAT clips for the logit comparison (plus assets/audio-sampels)

# Test-set PER. The full base-v1 test set is ~33k utterances (days of CPU at batch size 1 for
# 5 runs), so score a fixed random subset. 0 skips the test set; None means the full set.
TEST_SUBSET = 300

# Which bundles to check (None = all four: litert/fp32, litert/int8, onnx/fp32, onnx/int8).
BUNDLES = None

In [ ]:
# Hugging Face token from Colab secrets, checked before the long install.
import os

from huggingface_hub import model_info

HF_TOKEN = None
try:
    from google.colab import userdata

    for name in ("HF_TOKEN", "HF_TOKEN_MUALEM"):
        try:
            HF_TOKEN = userdata.get(name).strip()
            print(f"Using the Colab secret {name}")
            break
        except Exception:
            pass
except ImportError:
    pass
if not HF_TOKEN:
    from getpass import getpass

    HF_TOKEN = getpass("Hugging Face token: ").strip()

os.environ["HF_TOKEN"] = HF_TOKEN  # the exporter, validator and datasets all read this
try:
    model_info(MODEL_ID, token=HF_TOKEN)
except Exception as error:
    raise SystemExit(
        f"This token cannot read {MODEL_ID}: {error}\n"
        "Use a token whose account has been granted access to the repo."
    )
print(f"Token can read {MODEL_ID}")

In [ ]:
# Where outputs go.
from pathlib import Path

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    OUT = Path("/content/drive/MyDrive/fastconformer_export") / MODEL_ID.split("/")[-1]
else:
    OUT = Path("/content/exports") / MODEL_ID.split("/")[-1]
OUT.mkdir(parents=True, exist_ok=True)
print("Outputs:", OUT)

In [ ]:
# Get the code. An existing checkout is hard-reset to the branch, so push changes before re-running.
import subprocess

REPO = Path("/content/prepare-quran-dataset")


def run(command, cwd=REPO, check=True):
    """Run a shell command, streaming its output; stop "Run all" if it fails."""
    print(f"$ {command}", flush=True)
    process = subprocess.Popen(
        command, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end="", flush=True)
    code = process.wait()
    if check and code != 0:
        raise RuntimeError(f"Command failed with exit code {code}: {command}")
    return code


if REPO.exists():
    run(f"git fetch --depth 1 origin {BRANCH} && git reset --hard FETCH_HEAD")
else:
    run(f"git clone -b {BRANCH} --depth 1 {REPO_URL} {REPO}", cwd="/content")
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
print("Commit:", COMMIT)

In [ ]:
# Install. The repo needs Python 3.14, which Colab doesn't have, so uv downloads it and installs
# exactly the versions in uv.lock into /content/prepare-quran-dataset/.venv. Colab's own torch is
# not used anywhere below.
run("pip install -q -U uv", cwd="/content")
run("uv sync --locked --extra annotate --extra nemo --extra litert --extra onnx --extra test")
run(
    "uv run python -c \""
    "import sys, torch, nemo, litert_torch, onnxruntime, ai_edge_litert, importlib.metadata as m; "
    "print('python', sys.version.split()[0]); print('torch', torch.__version__); "
    "print('nemo', nemo.__version__); "
    "[print(p, m.version(p)) for p in ('litert-torch', 'ai-edge-litert', 'ai-edge-quantizer', 'onnxruntime', 'onnx')]\""
)

## 1. Unit tests (gates 1, 2, 3, 6, 7)

These use a tiny random model and synthetic audio, so they need no checkpoint. Expect everything to pass:
the mel tests compare against the NeMo installed here (2.1.0). If NeMo were upgraded, its STFT padding
would change and the mel tests would fail on purpose.

In [ ]:
run(
    "uv run pytest -v -p no:cacheprovider "
    "tests/test_fastconformer_export_mel.py "
    "tests/test_fastconformer_export_graphs.py "
    "tests/test_fastconformer_mask_golden.py "
    "tests/test_ctc_decoding.py "
    "tests/test_streaming_processor.py"
)

## 2. Export the real checkpoint

Skipped if `OUT` already holds an export of the same model from the same commit.

In [ ]:
import json

stamp_path = OUT / "notebook_export_stamp.json"
stamp = {"model_id": MODEL_ID, "commit": COMMIT}
if (OUT / "manifest.json").exists() and stamp_path.exists() and json.loads(stamp_path.read_text()) == stamp:
    print("Reusing the existing export in", OUT)
else:
    run(
        f"uv run python -m prepare_quran_dataset.fastconformer_export.export "
        f"--model-id {MODEL_ID} --output-dir '{OUT}'"
    )
    stamp_path.write_text(json.dumps(stamp))

In [ ]:
# What was exported: sizes, Flex ops (must be none), XNNPACK partitions, int8 tensors.
manifest = json.loads((OUT / "manifest.json").read_text())
print("Direct torch.stft attempt:", manifest.get("mel_direct_stft_attempt"))
print("Quantization:", json.dumps(manifest.get("quantization"), indent=1))
print()
print(f"{'graph':28} {'MB':>8} {'flex':>5} {'xnnpack parts':>14} {'int8 tensors':>13}")
for graph, report in manifest["reports"]["litert"].items():
    print(
        f"litert/{graph:21} {report['bytes'] / 1e6:8.1f} {len(report['flex_ops']):5d} "
        f"{report['xnnpack_partitions']:14d} {report['tensor_types'].get('int8', 0):13d}"
    )
for graph, report in manifest["reports"]["onnx"].items():
    quantized = sum(report["ops"].get(op, 0) for op in ("MatMulInteger", "ConvInteger", "DynamicQuantizeLinear"))
    print(f"onnx/{graph:23} {report['bytes'] / 1e6:8.1f} {'':>5} {'':>14} {quantized:13d}")

## 3. Fidelity (gate 4)

Every bundle runs from raw PCM through all four graphs and is compared, logit by logit, with
`infer_fastconformer_streaming` on the audio in `assets/audio-sampels` plus `NUM_QDAT_FIDELITY` QDAT clips.
fp32 must stay within 1e-3 with identical argmax. int8 only has to be finite here; PER judges it.

In [ ]:
bundles_flag = f" --bundles {','.join(BUNDLES)}" if BUNDLES else ""
fidelity_code = run(
    f"uv run python -m prepare_quran_dataset.fastconformer_export.validate "
    f"--model-id {MODEL_ID} --artifacts '{OUT}' --checks fidelity "
    f"--num-qdat-utterances {NUM_QDAT_FIDELITY}{bundles_flag}",
    check=False,
)
print("\nFidelity", "PASSED" if fidelity_code == 0 else "FAILED (see the summary below)")

## 4. PER (gate 5)

QDAT runs in full; the test set is a fixed random subset of `TEST_SUBSET` utterances (the first run
downloads and phonetizes the 3 test moshafs, ~10 GB). Each finished run is saved under
`validation/per/`, so if Colab disconnects, re-run all and it continues from where it stopped.

In [ ]:
if TEST_SUBSET == 0:
    testset_flag = " --skip-testset"
elif TEST_SUBSET is None:
    testset_flag = ""
else:
    testset_flag = f" --max-test-utterances {TEST_SUBSET}"
per_code = run(
    f"uv run python -m prepare_quran_dataset.fastconformer_export.validate "
    f"--model-id {MODEL_ID} --artifacts '{OUT}' --checks per{testset_flag}{bundles_flag}",
    check=False,
)
print("\nPER", "PASSED" if per_code == 0 else "FAILED (see the summary below)")

## 5. Summary

In [ ]:
validation = OUT / "validation"
fidelity_path = validation / "report_fidelity.json"
per_path = validation / "report_per.json"

if fidelity_path.exists():
    fidelity = json.loads(fidelity_path.read_text())["fidelity"]
    print(f"Fidelity over {fidelity['utterances']} utterances (fp32 limit 1e-3, identical argmax)")
    print(f"  {'bundle':14} {'max abs err':>12} {'argmax agree':>13} {'non-finite':>11}  result")
    for bundle, s in fidelity["bundles"].items():
        print(
            f"  {bundle:14} {s['max_abs_error']:12.2e} {s['argmax_agreement']:13.4%} "
            f"{s['non_finite']:11d}  {'PASS' if s['passed'] else 'FAIL'}"
        )
else:
    print("No fidelity report:", fidelity_path)

print()
if per_path.exists():
    per = json.loads(per_path.read_text())["per"]
    print("PER settings:", per["settings"])
    keys = ["qdat_per_phonemes", "qdat_average_per", "test_per_phonemes", "test_average_per"]
    print(f"  {'run':14}" + "".join(f"{k:>20}" for k in keys) + "  result")
    for name, values in per["per"].items():
        gate = per["gates"].get(name)
        result = "reference" if gate is None else ("PASS" if gate["passed"] else "FAIL")
        cells = "".join(
            f"{values.get(k):20.5f}" if values.get(k) is not None else f"{'-':>20}" for k in keys
        )
        print(f"  {name:14}{cells}  {result}")
    print("\n  Limits: fp32 |Δ| ≤ 2e-4 on every metric; int8 QDAT phoneme PER at most 0.005 worse.")
else:
    print("No PER report:", per_path)

print(f"\nSend back: {validation} (at least report_fidelity.json and report_per.json)")